# NovaCart — Silver Orders Transformation

In [0]:
from pyspark.sql import functions as F

## 1. Define Storage Paths

In [0]:
BRONZE_ORDERS_PATH = (
    "abfss://bronze@stnovacartdev.dfs.core.windows.net/"
    "olist/orders"
)

SILVER_ORDERS_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/orders"
)

QUARANTINE_ORDERS_PATH = (
    "abfss://quarantine@stnovacartdev.dfs.core.windows.net/"
    "olist/orders"
)

print(f"Bronze path: {BRONZE_ORDERS_PATH}")
print(f"Silver path: {SILVER_ORDERS_PATH}")
print(f"Quarantine path: {QUARANTINE_ORDERS_PATH}")

## 2. Read Bronze Orders Data

In [0]:
orders_bronze_df = (
    spark.read
    .format("delta")
    .load(BRONZE_ORDERS_PATH)
)

bronze_row_count = orders_bronze_df.count()

print("Bronze orders loaded successfully.")
print(f"Bronze row count: {bronze_row_count}")

orders_bronze_df.printSchema()
display(orders_bronze_df.limit(10))

## 3. Validate Required Columns

In [0]:
required_columns = [
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "_source_file",
    "_ingestion_timestamp",
    "_batch_id",
]

missing_columns = [
    column
    for column in required_columns
    if column not in orders_bronze_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required Bronze columns: {missing_columns}"
    )

print("Required-column validation passed.")

## 4. Profile Order Status Values

In [0]:
display(
    orders_bronze_df
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

## 5. Profile Missing Values

In [0]:
orders_profile_df = orders_bronze_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        (
            F.col("order_id").isNull()
            | (F.trim(F.col("order_id")) == "")
        ).cast("int")
    ).alias("invalid_order_id"),

    F.sum(
        (
            F.col("customer_id").isNull()
            | (F.trim(F.col("customer_id")) == "")
        ).cast("int")
    ).alias("invalid_customer_id"),

    F.sum(
        (
            F.col("order_status").isNull()
            | (F.trim(F.col("order_status")) == "")
        ).cast("int")
    ).alias("invalid_order_status"),

    F.sum(
        F.col("order_purchase_timestamp").isNull().cast("int")
    ).alias("missing_purchase_timestamp"),

    F.sum(
        F.col("order_approved_at").isNull().cast("int")
    ).alias("missing_approval_timestamp"),

    F.sum(
        F.col("order_delivered_carrier_date").isNull().cast("int")
    ).alias("missing_carrier_delivery_timestamp"),

    F.sum(
        F.col("order_delivered_customer_date").isNull().cast("int")
    ).alias("missing_customer_delivery_timestamp"),

    F.sum(
        F.col("order_estimated_delivery_date").isNull().cast("int")
    ).alias("missing_estimated_delivery_timestamp"),
)

display(orders_profile_df)

## 6. Check Duplicate Order IDs

In [0]:
duplicate_order_ids_df = (
    orders_bronze_df
    .groupBy("order_id")
    .count()
    .filter(
        F.col("order_id").isNotNull()
        & (F.col("count") > 1)
    )
)

duplicate_order_id_count = duplicate_order_ids_df.count()

print(
    f"Number of order_id values appearing more than once: "
    f"{duplicate_order_id_count}"
)

display(duplicate_order_ids_df.limit(20))

## 7. Define Allowed Order Statuses

In [0]:
allowed_order_statuses = [
    "approved",
    "canceled",
    "created",
    "delivered",
    "invoiced",
    "processing",
    "shipped",
    "unavailable",
]

## 8. Clean and Standardize Order Fields

In [0]:
orders_cleaned_df = (
    orders_bronze_df
    .withColumn(
        "order_id",
        F.trim(F.col("order_id"))
    )
    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id"))
    )
    .withColumn(
        "order_status",
        F.lower(F.trim(F.col("order_status")))
    )
)

## 9. Profile Invalid Statuses and Date Chronology

In [0]:
orders_quality_profile_df = orders_cleaned_df.agg(
    F.sum(
        (
            F.col("order_id").isNull()
            | (F.col("order_id") == "")
        ).cast("int")
    ).alias("invalid_order_id"),

    F.sum(
        (
            F.col("customer_id").isNull()
            | (F.col("customer_id") == "")
        ).cast("int")
    ).alias("invalid_customer_id"),

    F.sum(
        (
            F.col("order_status").isNull()
            | (F.col("order_status") == "")
            | ~F.col("order_status").isin(allowed_order_statuses)
        ).cast("int")
    ).alias("invalid_order_status"),

    F.sum(
        F.col("order_purchase_timestamp").isNull().cast("int")
    ).alias("missing_purchase_timestamp"),

    F.sum(
        (
            F.col("order_approved_at").isNotNull()
            & F.col("order_purchase_timestamp").isNotNull()
            & (
                F.col("order_approved_at")
                < F.col("order_purchase_timestamp")
            )
        ).cast("int")
    ).alias("approval_before_purchase"),

    F.sum(
        (
            F.col("order_delivered_carrier_date").isNotNull()
            & F.col("order_purchase_timestamp").isNotNull()
            & (
                F.col("order_delivered_carrier_date")
                < F.col("order_purchase_timestamp")
            )
        ).cast("int")
    ).alias("carrier_delivery_before_purchase"),

    F.sum(
        (
            F.col("order_delivered_customer_date").isNotNull()
            & F.col("order_delivered_carrier_date").isNotNull()
            & (
                F.col("order_delivered_customer_date")
                < F.col("order_delivered_carrier_date")
            )
        ).cast("int")
    ).alias("customer_delivery_before_carrier"),

    F.sum(
        (
            F.col("order_estimated_delivery_date").isNotNull()
            & F.col("order_purchase_timestamp").isNotNull()
            & (
                F.col("order_estimated_delivery_date")
                < F.col("order_purchase_timestamp")
            )
        ).cast("int")
    ).alias("estimated_delivery_before_purchase")
)

display(orders_quality_profile_df)

## 10. Define Order Validation Rules

In [0]:
invalid_order_id_condition = (
    F.col("order_id").isNull()
    | (F.col("order_id") == "")
)

invalid_customer_id_condition = (
    F.col("customer_id").isNull()
    | (F.col("customer_id") == "")
)

invalid_order_status_condition = (
    F.col("order_status").isNull()
    | (F.col("order_status") == "")
    | ~F.col("order_status").isin(allowed_order_statuses)
)

missing_purchase_timestamp_condition = (
    F.col("order_purchase_timestamp").isNull()
)

approval_before_purchase_condition = (
    F.col("order_approved_at").isNotNull()
    & F.col("order_purchase_timestamp").isNotNull()
    & (
        F.col("order_approved_at")
        < F.col("order_purchase_timestamp")
    )
)

carrier_before_purchase_condition = (
    F.col("order_delivered_carrier_date").isNotNull()
    & F.col("order_purchase_timestamp").isNotNull()
    & (
        F.col("order_delivered_carrier_date")
        < F.col("order_purchase_timestamp")
    )
)

customer_before_carrier_condition = (
    F.col("order_delivered_customer_date").isNotNull()
    & F.col("order_delivered_carrier_date").isNotNull()
    & (
        F.col("order_delivered_customer_date")
        < F.col("order_delivered_carrier_date")
    )
)

estimated_before_purchase_condition = (
    F.col("order_estimated_delivery_date").isNotNull()
    & F.col("order_purchase_timestamp").isNotNull()
    & (
        F.col("order_estimated_delivery_date")
        < F.col("order_purchase_timestamp")
    )
)

## 11. Assign Order Rejection Reasons

In [0]:
orders_validated_df = orders_cleaned_df.withColumn(
    "_rejection_reason",

    F.when(
        invalid_order_id_condition,
        F.lit("MISSING_ORDER_ID")
    )
    .when(
        invalid_customer_id_condition,
        F.lit("MISSING_CUSTOMER_ID")
    )
    .when(
        invalid_order_status_condition,
        F.lit("INVALID_ORDER_STATUS")
    )
    .when(
        missing_purchase_timestamp_condition,
        F.lit("MISSING_PURCHASE_TIMESTAMP")
    )
    .when(
        approval_before_purchase_condition,
        F.lit("APPROVAL_BEFORE_PURCHASE")
    )
    .when(
        carrier_before_purchase_condition,
        F.lit("CARRIER_DELIVERY_BEFORE_PURCHASE")
    )
    .when(
        customer_before_carrier_condition,
        F.lit("CUSTOMER_DELIVERY_BEFORE_CARRIER")
    )
    .when(
        estimated_before_purchase_condition,
        F.lit("ESTIMATED_DELIVERY_BEFORE_PURCHASE")
    )
    .otherwise(F.lit(None))
)

## 12. Review Order Validation Results

In [0]:
display(
    orders_validated_df
    .groupBy("_rejection_reason")
    .count()
    .orderBy("_rejection_reason")
)

## 13. Split Valid and Invalid Orders

In [0]:
orders_valid_df = (
    orders_validated_df
    .filter(F.col("_rejection_reason").isNull())
    .drop("_rejection_reason")
)

orders_quarantine_df = (
    orders_validated_df
    .filter(F.col("_rejection_reason").isNotNull())
)

## 14. Derive Delivery Metrics

In [0]:
orders_silver_df = (
    orders_valid_df

    .withColumn(
        "delivery_days",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_purchase_timestamp")
        )
    )

    .withColumn(
        "estimated_delivery_days",
        F.datediff(
            F.col("order_estimated_delivery_date"),
            F.col("order_purchase_timestamp")
        )
    )

    .withColumn(
        "delivery_delay_days",
        F.when(
            F.col("order_delivered_customer_date").isNotNull()
            & F.col("order_estimated_delivery_date").isNotNull(),
            F.datediff(
                F.col("order_delivered_customer_date"),
                F.col("order_estimated_delivery_date")
            )
        )
    )

    .withColumn(
        "is_delayed",
        F.when(
            F.col("order_delivered_customer_date").isNull()
            | F.col("order_estimated_delivery_date").isNull(),
            F.lit(None).cast("boolean")
        ).otherwise(
            F.col("order_delivered_customer_date")
            > F.col("order_estimated_delivery_date")
        )
    )

    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

## 15. Add Quarantine Metadata

In [0]:
orders_quarantine_df = (
    orders_quarantine_df
    .withColumn(
        "_quarantined_at",
        F.current_timestamp()
    )
    .withColumn(
        "_source_dataset",
        F.lit("orders")
    )
)

## 16. Count Silver and Quarantine Orders

In [0]:
valid_row_count = orders_silver_df.count()
quarantine_row_count = orders_quarantine_df.count()

print(f"Valid Silver rows: {valid_row_count}")
print(f"Quarantined rows: {quarantine_row_count}")
print(f"Bronze input rows: {bronze_row_count}")

## 17. Validate Row-Count Reconciliation

In [0]:
if valid_row_count + quarantine_row_count != bronze_row_count:
    raise ValueError(
        "Row-count validation failed: "
        "Silver rows + quarantine rows do not equal Bronze input rows."
    )

print("Row-count validation passed.")

## 18. Inspect Derived Order Fields

In [0]:
display(
    orders_silver_df.select(
        "order_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "estimated_delivery_days",
        "delivery_delay_days",
        "is_delayed"
    ).limit(20)
)

## 19. Write Valid Orders to Silver

In [0]:
(
    orders_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_ORDERS_PATH)
)

print("Silver orders written successfully.")

## 20. Write Invalid Orders to Quarantine

In [0]:
(
    orders_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(QUARANTINE_ORDERS_PATH)
)

print("Orders quarantine output written successfully.")

## 21. Read Written Delta Outputs

In [0]:
orders_silver_written_df = (
    spark.read
    .format("delta")
    .load(SILVER_ORDERS_PATH)
)

orders_quarantine_written_df = (
    spark.read
    .format("delta")
    .load(QUARANTINE_ORDERS_PATH)
)

silver_written_count = orders_silver_written_df.count()
quarantine_written_count = orders_quarantine_written_df.count()

print(f"Written Silver rows: {silver_written_count}")
print(f"Written quarantine rows: {quarantine_written_count}")

## 22. Validate Written Outputs

In [0]:
if silver_written_count != valid_row_count:
    raise ValueError(
        "Silver write validation failed: "
        f"expected {valid_row_count}, wrote {silver_written_count}."
    )

if quarantine_written_count != quarantine_row_count:
    raise ValueError(
        "Quarantine write validation failed: "
        f"expected {quarantine_row_count}, wrote "
        f"{quarantine_written_count}."
    )

if silver_written_count + quarantine_written_count != bronze_row_count:
    raise ValueError(
        "Final reconciliation failed: "
        "Silver + quarantine does not equal Bronze."
    )

print("Silver orders pipeline completed successfully.")
print("Final row-count validation passed.")

## 23. Inspect Final Silver Orders Dataset

In [0]:
orders_silver_written_df.printSchema()

display(
    orders_silver_written_df.select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "estimated_delivery_days",
        "delivery_delay_days",
        "is_delayed",
        "_silver_processed_at"
    ).limit(20)
)